# 🐟 Aqua Bot AI — พี่ปลา
**AI Companion ผู้ช่วยเลี้ยงปลาเศรษฐกิจ**

ระบบ RAG ที่ใช้ข้อมูลจาก `Aqua Bot_Cleaned_ReadyForModel`  
ขับเคลื่อนด้วย **GROQ (LLaMA-3)** + **FAISS** +  **FastAPI + ngrok**

---
### 🔑 Secrets ที่ต้องตั้งใน Google Colab → Secrets:
| Key | หาได้จาก |
|-----|----------|
| `GROQ_TOKEN` | https://console.groq.com → API Keys |
| `LC_TOKEN` | https://smith.langchain.com → Settings → API Keys |
| `NGROK_TOKEN` | https://dashboard.ngrok.com → Your Authtoken |




## 1 : ติดตั้ง Dependencies

In [1]:
# ติดตั้งแพ็คเกจที่จำเป็น
!pip install -q -U langchain langchain-community langchain-core langchain_text_splitters
!pip install -q langchain_chroma langchain-huggingface langchain-groq langchain-classic
!pip install -q faiss-cpu sentence_transformers
!pip install -q rapidfuzz python-Levenshtein
!pip install -q fastapi uvicorn pyngrok nest_asyncio
!pip install -q python-jose
!pip install -q gdown
print('✅ ติดตั้งเสร็จสิ้น!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━

## 2 : Restart Runtime (รันครั้งเดียวหลังติดตั้ง)

In [ ]:
# รันเซลล์นี้เพื่อ restart kernel หลังติดตั้ง packages
# จากนั้นข้ามไปรัน Step 3 ต่อได้เลย
import os
os._exit(00)

## 3 : Import Libraries & ตั้งค่า API Keys

In [2]:
import sys
import os
import pandas as pd
import re
import torch
import unicodedata
from difflib import SequenceMatcher, get_close_matches
from rapidfuzz import process
from Levenshtein import ratio
import time

from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# ── ดึง token จาก Colab Secrets ──
langchain_token = userdata.get('LC_TOKEN')
groq_token      = userdata.get('GROQ_TOKEN')
ngrok_token     = userdata.get('NGROK_TOKEN')
pinecone_token = userdata.get('PINECONE_API_KEY')

# ── ตั้งค่า API Keys ──
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_API_KEY']    = langchain_token
os.environ['GROQ_API_KEY']         = groq_token
os.environ['PINECONE_API_KEY'] = pinecone_token

print('✅ ตั้งค่า API Keys เรียบร้อย!')

✅ ตั้งค่า API Keys เรียบร้อย!




## 4 : โหลด CSV & สร้าง FAISS Vector Store



In [3]:
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn
from starlette.middleware.cors import CORSMiddleware # Import CORSMiddleware

nest_asyncio.apply()

app = FastAPI()

# Add CORS Middleware here
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Allows all origins, you might want to restrict this in production
    allow_credentials=True, # Allow credentials (e.g., cookies, authorization headers)
    allow_methods=["*"],  # Allows all methods (GET, POST, PUT, DELETE, etc.)
    allow_headers=["*"],  # Allows all headers
)

class ChatRequest(BaseModel):
    message: str

@app.post("/chat")
async def chat_endpoint(request: ChatRequest):
    try:
        response = handle_query(request.message)
        return {"response": response}
    except Exception as e:
        return {"response": f"เกิดข้อผิดพลาด: {str(e)}"}


In [4]:
import gdown
import os

KB_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1vP3XJoGaXBullirWfxg0Az5k1nvfkSII'
LOCAL_KB_FOLDER = 'knowledge_manuals'

# สร้างโฟลเดอร์ถ้ายังไม่มี
if not os.path.exists(LOCAL_KB_FOLDER):
    os.makedirs(LOCAL_KB_FOLDER)

print(f'⬇️ กำลังดาวน์โหลดโฟลเดอร์ Knowledge Base จาก Google Drive ไปที่ \'{LOCAL_KB_FOLDER}\'...')
# [FIX]: OSErorr: [Errno 36] File name too long can occur if original filenames in Google Drive are too long.
#       Please shorten filenames in your Google Drive folder to resolve this issue.
gdown.download_folder(KB_DRIVE_FOLDER_URL, output=LOCAL_KB_FOLDER, quiet=False, use_cookies=False)
print(f'✅ ดาวน์โหลดโฟลเดอร์ \'{LOCAL_KB_FOLDER}\' เสร็จสิ้น!')

# อัปเดตตัวแปร KB_FOLDER ให้ชี้ไปที่โฟลเดอร์ที่ดาวน์โหลดมา
KB_FOLDER = LOCAL_KB_FOLDER

⬇️ กำลังดาวน์โหลดโฟลเดอร์ Knowledge Base จาก Google Drive ไปที่ 'knowledge_manuals'...


Retrieving folder contents


Processing file 1vVa0-Z_04KBUXqoXXuZJpxLHp3FsA1f0 Aqua Bot_Cleaned_ReadyForModel.csv
Processing file 1R44jEpxXuS19H0TF0Z83e36E1Ed6F-bB การทำก้อนบำบัดน้ำเสีย (ดังโหงะ) กระทรวงเกษรและสหกรณ์.csv
Processing file 1bPrnjZcOE3Ut4s3st3OxQWKpL7qTzylw การบําบัดน้ําทิ้งจากฟาร์มเพาะเลี้ยงสัตว์น้ําด้วยพืชน้ําและสาหร่าย กรมประมง.csv
Processing file 1r-O12Hldqyj0SJIrLsJa9hnJxRvhXNgS การป้องกันและกำจัดโรคปลา.csv
Processing file 171T5wKDZAUMsoN85zoTDGf3CH7AQhcQo การเพาะเลี้ยงปลาดุกบิ๊กอุย.csv
Processing file 1ZvxiMm2wtUvfd7oUK9ToJRVKK4j2jVM8 การเพาะเลี้ยงปลาตะเพียน.csv
Processing file 11SjUWozBl9JEEa28-r1E9fsjEIU4gyb4 การสังเกตอาการการป้องกน้ำและการรักษา.csv
Processing file 1RZKIXo2PC-QnTtD3xSALcJZiZGqNHJfG ข้อกำหนด GAP ประมง.csv
Processing file 1JB1tGrE9cpbtseNmtvxo4KlrRY3_q4Gj ค่มือการผลติและปรับปรุงพนัธ์ปลากดแก้ว.csv
Processing file 11K8D0MJ1dRXNIIrDvhuIqpjJ9mz284Vj คู่มือการเพิ่มประสิทธิภาพและการลดต้นทุนการเลี้ยงปลานิล.csv
Processing file 1RpfhknLkBpQiDszXEvugrgUm6cF2vb13 ปริมาณและมูลค่าผลผลิตสัตว์

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1vVa0-Z_04KBUXqoXXuZJpxLHp3FsA1f0
To: /content/knowledge_manuals/Aqua Bot_Cleaned_ReadyForModel.csv
100%|██████████| 475k/475k [00:00<00:00, 6.32MB/s]
Downloading...
From: https://drive.google.com/uc?id=1R44jEpxXuS19H0TF0Z83e36E1Ed6F-bB
To: /content/knowledge_manuals/การทำก้อนบำบัดน้ำเสีย (ดังโหงะ) กระทรวงเกษรและสหกรณ์.csv
100%|██████████| 4.09k/4.09k [00:00<00:00, 8.07MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bPrnjZcOE3Ut4s3st3OxQWKpL7qTzylw
To: /content/knowledge_manuals/การบําบัดน้ําทิ้งจากฟาร์มเพาะเลี้ยงสัตว์น้ําด้วยพืชน้ําและสาหร่าย กรมประมง.csv
100%|██████████| 12.6k/12.6k [00:00<00:00, 17.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1r-O12Hldqyj0SJIrLsJa9hnJxRvhXNgS
To: /content/knowledge_manuals/การป้องกันและกำจัดโรคปลา.csv
100%|██████████| 116k/116k [00:00<00:00, 2.96MB/s]
Downloading...
From

✅ ดาวน์โหลดโฟลเดอร์ 'knowledge_manuals' เสร็จสิ้น!



Download completed


In [5]:
import gdown
import os

# ── 1. โหลดข้อมูล CSV ──
# ใช้ LOCAL_KB_FOLDER ที่ถูกกำหนดไว้แล้ว
CSV_PATH = os.path.join(LOCAL_KB_FOLDER, 'Aqua Bot_Cleaned_ReadyForModel.csv')
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
df = df.fillna('ไม่มีข้อมูล')
print(f'📊 โหลด {len(df)} แถวจาก CSV')
print('คอลัมน์ทั้งหมด:', list(df.columns))

# ════════════════════════════════════════════════════
# 📝 row_to_text
# ════════════════════════════════════════════════════
def row_to_text(row):
    y_full = str(row.get('ปี', ''))
    y_short = y_full[-2:] if len(y_full) == 4 else ''

    text = f"ปี:{y_full}({y_short})|ปลา:{row.get('ชนิดสัตว์น้ำ','')}|จ.{row.get('จังหวัด','')}({row.get('ภาค','')})"

    if pd.notna(row.get('จำนวนฟาร์ม')): text += f"|ฟาร์ม:{row['จำนวนฟาร์ม']}แห่ง"
    if pd.notna(row.get('ปริมาณ')): text += f"|ปริมาณ:{row['ปริมาณ']}ตัน"

    if pd.notna(row.get('ผลผลิตต่อไร่(ตัน)')):
        try: text += f"|ผลผลิต:{round(float(row['ผลผลิตต่อไร่(ตัน)']), 2)}ตัน/ไร่"
        except: pass

    if pd.notna(row.get('ราคาเฉลี่ย(พันบาท/ตัน)')):
        try: text += f"|ราคา:{round(float(row['ราคาเฉลี่ย(พันบาท/ตัน)']), 2)}พันบาท/ตัน(หรือบาท/กก.)"
        except: pass

    return text.strip()
# ── 2. สร้าง Document Objects ──
from langchain_core.documents import Document

docs = []
for idx, row in df.iterrows():
    page_content = row_to_text(row)
    metadata = {col: str(row[col]) for col in df.columns}
    docs.append(Document(page_content=page_content, metadata=metadata))

print(f'📄 สร้าง Document สำเร็จ {len(docs)} รายการ')
print('ตัวอย่าง Document ใบแรก:', docs[0].page_content)

# ── 3. สร้าง FAISS Vector Store ──
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print('🧠 กำลัง Embed ข้อความและสร้าง FAISS Index (อาจใช้เวลาสักครู่)...')
vectorstore = FAISS.from_documents(docs, embeddings)
print('✅ สร้าง Vector Store สำเร็จ!')

📊 โหลด 2200 แถวจาก CSV
คอลัมน์ทั้งหมด: ['_id', 'ปี', 'ประเภทการเพาะเลี้ยง', 'จังหวัด', 'ภาค', 'ประเภทการเลี้ยง', 'ชนิดสัตว์น้ำ', 'จำนวนฟาร์ม', 'เนื้อที่เลี้ยง', 'ปริมาณ', 'มูลค่า', 'ผลผลิตต่อไร่(ตัน)', 'ราคาเฉลี่ย(พันบาท/ตัน)']
📄 สร้าง Document สำเร็จ 2200 รายการ
ตัวอย่าง Document ใบแรก: ปี:2563(63)|ปลา:ปลานิล|จ.กำแพงเพชร(ภาคกลาง)|ฟาร์ม:3460แห่ง|ปริมาณ:1763.0ตัน|ผลผลิต:0.56ตัน/ไร่|ราคา:47.93พันบาท/ตัน(หรือบาท/กก.)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🧠 กำลัง Embed ข้อความและสร้าง FAISS Index (อาจใช้เวลาสักครู่)...
✅ สร้าง Vector Store สำเร็จ!


In [6]:
# รันเซลล์นี้เพื่อเพิ่ม KB เข้า vectorstore ที่มีอยู่
import os
import glob
from langchain_core.documents import Document

# KB_FOLDER ถูกตั้งค่าในเซลล์ก่อนหน้า 'download_kb_folder'
# ถ้าคุณรันเซลล์นี้แยกต่างหาก ตรวจสอบให้แน่ใจว่า KB_FOLDER ชี้ไปที่โฟลเดอร์ที่ดาวน์โหลดมาแล้ว
# เช่น KB_FOLDER = 'knowledge_manuals'

kb_docs = []

# ค้นหาไฟล์ CSV ทั้งหมดในโฟลเดอร์ที่ดาวน์โหลดมา
manual_files = glob.glob(os.path.join(KB_FOLDER, '*.csv'))

print(f'📂 พบไฟล์ KB CSV ทั้งหมด {len(manual_files)} ไฟล์ใน \'{KB_FOLDER}\'')

for fpath in sorted(manual_files):
    fname = os.path.basename(fpath)
    try:
        kb_df = pd.read_csv(fpath, encoding='utf-8-sig')
        if not {'Topic', 'Content'}.issubset(kb_df.columns):
            print(f'❌ {fname}: ไม่มีคอลัมน์ที่จำเป็น (Topic, Content) หรือชื่อไม่ตรง')
            continue
        for _, row in kb_df.iterrows():
            content = str(row.get('Content', '')).strip()
            topic   = str(row.get('Topic', ''))
            if content:
                kb_docs.append(Document(
                    page_content=f"[{topic}] {content}",
                    metadata={'source': fname, 'topic': topic, 'type': 'knowledge'}
                ))
        print(f'✅ {fname}: {len(kb_df)} chunks')
    except Exception as e:
        print(f'❌ {fname}: {e}')

print(f'\n📚 KB docs: {len(kb_docs)} chunks')

# เพิ่มเข้า vectorstore ที่มีอยู่แล้ว (ไม่ต้อง rebuild ใหม่ทั้งหมด)
vectorstore.add_documents(kb_docs)
print('✅ เพิ่ม KB เข้า vectorstore สำเร็จ!')

# ตรวจสอบ
results = vectorstore.similarity_search("ก้อนบำบัดน้ำเสีย", k=3)
for r in results:
    print(r.metadata.get('type', 'ไม่มี type'), '|', r.page_content[:100])
all_docs = docs + kb_docs
print(f"📦 all_docs รวม: {len(all_docs)} รายการ")

📂 พบไฟล์ KB CSV ทั้งหมด 16 ไฟล์ใน 'knowledge_manuals'
❌ Aqua Bot_Cleaned_ReadyForModel.csv: ไม่มีคอลัมน์ที่จำเป็น (Topic, Content) หรือชื่อไม่ตรง
✅ การทำก้อนบำบัดน้ำเสีย (ดังโหงะ) กระทรวงเกษรและสหกรณ์.csv: 5 chunks
✅ การบําบัดน้ําทิ้งจากฟาร์มเพาะเลี้ยงสัตว์น้ําด้วยพืชน้ําและสาหร่าย กรมประมง.csv: 3 chunks
✅ การป้องกันและกำจัดโรคปลา.csv: 59 chunks
✅ การสังเกตอาการการป้องกน้ำและการรักษา.csv: 86 chunks
✅ การเพาะเลี้ยงปลาดุกบิ๊กอุย.csv: 28 chunks
✅ การเพาะเลี้ยงปลาตะเพียน.csv: 21 chunks
✅ ข้อกำหนด GAP ประมง.csv: 9 chunks
✅ คู่มือการเพิ่มประสิทธิภาพและการลดต้นทุนการเลี้ยงปลานิล.csv: 27 chunks
✅ ค่มือการผลติและปรับปรุงพนัธ์ปลากดแก้ว.csv: 114 chunks
❌ ปริมาณและมูลค่าผลผลิตสัตว์น้ำจากการเพาะเลี้ยงสัตว์น้ำจืด 2568.csv: ไม่มีคอลัมน์ที่จำเป็น (Topic, Content) หรือชื่อไม่ตรง
✅ ผลของพืชสมุนไพรต่อภูมิคุ้มกันและการเจริญเติบโตของปลา BUU.csv: 26 chunks
✅ มาตรฐานGAP.csv: 17 chunks
✅ มาตรฐานการปฏิบัติทางการเพาะเลี้ยงสัตว์น้ำที่ดี KU.csv: 36 chunks
✅ วิธีทำปุ๋ยน้ำหมักชีวภาพ พด.2 กรมพัฒนาที่ดิน.csv: 7 chunk

## 5 : ตั้งค่า LLM + Scope + Refusal Policy + Logic พี่ปลา

In [7]:
# ── ตั้งค่า LLM ──
# 💡 เปลี่ยนเป็น 8B ตามคำแนะนำเพื่อโควต้าที่เยอะขึ้น และตอบไวขึ้น
llm = ChatGroq(model='llama-3.1-8b-instant', max_tokens=500, temperature=0.1)

# ════════════════════════════════════════════════════
# 🔒 REFUSAL POLICY
# ════════════════════════════════════════════════════
OUT_OF_SCOPE_TRIGGERS = [
    'ปลาทู', 'แซลมอน', 'ปลาทะเล', 'กุ้งแม่น้ำ', 'ปลากัด', 'ปลาคาร์พ', 'ปลาทอง', 'ปลาสวยงาม',
    'หมา', 'สุนัข', 'แมว', 'หมู', 'ไก่', 'วัว',
    'วิธีปลูกข้าว', 'ปลูกทุเรียน', 'ปลูกยาง',
]

def check_out_of_scope(query: str) -> str | None:
    query_lower = query.lower()
    for trigger in OUT_OF_SCOPE_TRIGGERS:
        if ' ' not in trigger:
            if re.search(r'\b' + re.escape(trigger) + r'\b', query_lower):
                return f"❌ ขออภัยครับ ณ ตอนนี้พี่ปลาไม่มีข้อมูลเกี่ยวกับ '{trigger}' ครับ"
        else:
            if trigger in query_lower:
                return f"❌ ขออภัยครับ ณ ตอนนี้พี่ปลาไม่มีข้อมูลเกี่ยวกับ '{trigger}' ครับ"
    return None

# ════════════════════════════════════════════════════
# 📐 SYSTEM PROMPT (ฉบับรีดไขมัน ประหยัด Token)
# ════════════════════════════════════════════════════
system_prompt = """คุณคือ "พี่ปลา" (Aqua Bot) ผู้ช่วย AI อัจฉริยะด้านการเพาะเลี้ยงสัตว์น้ำเศรษฐกิจ (ปลานิล, ปลาดุก, ปลาตะเพียน, ปลากดแก้ว)
บุคลิกภาพ: เป็นมิตร มืออาชีพ กระตือรือร้น เชี่ยวชาญและให้เกียรติเกษตรกร แทนตัวเองว่า "พี่ปลา" และลงท้ายด้วย "ครับ" เสมอ

ข้อมูลสำหรับใช้ตอบคำถาม (Context):
---------------------
{context}
---------------------

กฎเหล็กสำหรับการตอบคำถาม (ปฏิบัติตามอย่างเคร่งครัด):
1. 🛑 ห้ามคิดคำตอบเอง: ตอบคำถามโดยใช้ข้อมูลจาก [Context] เท่านั้น ห้ามเดาตัวเลขสถิติเด็ดขาด
2. 🎯 สั้น กระชับ ตรงประเด็น: สรุปใจความสำคัญมาตอบทันที ห้ามร่ายยาวน้ำท่วมทุ่ง พยายามตอบให้จบภายใน 3-5 บรรทัด หรือใช้ Bullet แบบสั้นๆ
3. 📊 สถิติแบบกระชับ: หากมีสถิติหลายจังหวัด ให้สรุปภาพรวม หรือยกตัวอย่างแค่ 1-2 จังหวัดที่เด่นที่สุด
4. 🧮 แปลงหน่วยเสมอ: เปลี่ยน "พันบาท/ตัน" เป็น "บาท/กิโลกรัม" อัตโนมัติ (เช่น 47.93 พันบาท/ตัน = 47.93 บาท/กก.)
5. 🛡️ กรณีไม่มีข้อมูล: หากหาใน Context ไม่เจอ ให้ตอบสั้นๆ ว่า "ขออภัยครับ จากฐานข้อมูลตอนนี้ พี่ปลายังไม่มีข้อมูลส่วนนี้ครับ"

ประวัติการสนทนาที่ผ่านมา (หากมี):
{chat_history}

คำถามล่าสุดจากเกษตรกร: {question}

คำตอบของพี่ปลา:"""

AQUA_PROMPT = PromptTemplate(
    template=system_prompt,
    input_variables=["context", "chat_history", "question"]
)

# ── สร้าง Vector Store แบบควบรวม ──
# ดึง all_docs จาก Cell ที่รันก่อนหน้านี้
print(f"📦 กำลังรวมข้อมูลเข้าสู่ Vector Store...")
main_vectorstore = FAISS.from_documents(all_docs, embeddings)

retriever = main_vectorstore.as_retriever(search_kwargs={"k": 5})

LIST_DATA_KEYWORDS = [
    'มีข้อมูลอะไรบ้าง', 'รายการปลาที่มี', 'ข้อมูลอะไรที่มี',
    'ลิสต์ข้อมูลทั้งหมด', 'แสดงไฟล์ข้อมูล', 'เอกสารทั้งหมด'
]

def get_available_content_summary() -> str:
    return "พี่ปลามีข้อมูลสถิติ (ปี 63-66) เกี่ยวกับปลา 4 ชนิด ได้แก่: ปลากดแก้ว, ปลาดุก, ปลาตะเพียน, ปลานิล ครับ และมีความรู้เรื่องการเลี้ยง โรค และการจัดการน้ำครับ"

def format_chat_history(history: list[tuple[str, str]]) -> str:
    if not history:
        return "ไม่มีประวัติการสนทนา"

    formatted_history = []
    for human_msg, ai_msg in history[-1:]:
        formatted_history.append(f"เกษตรกร: {human_msg}")
        # ตัดคำตอบเก่าของพี่ปลาให้เหลือแค่ 100 ตัวอักษร เพื่อเป็นบริบทเฉยๆ
        formatted_history.append(f"พี่ปลา: {ai_msg[:100]}...")
    return "\n".join(formatted_history)

def handle_query(query: str, chat_history: list[tuple[str, str]] = []) -> str:
    refusal = check_out_of_scope(query)
    if refusal:
        return refusal

    if any(kw in query for kw in LIST_DATA_KEYWORDS):
        return get_available_content_summary()

    search_query = re.sub(r'\b(59|60|61|62|63|64|65|66|67)\b', r'25\1', query)

    try:
        retrieved_docs = retriever.invoke(search_query)
        context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

        formatted_history_str = format_chat_history(chat_history)

        final_prompt = AQUA_PROMPT.format(
            context=context_text,
            chat_history=formatted_history_str,
            question=search_query
        )

        response = llm.invoke(final_prompt)
        return response.content

    except Exception as e:
        return f"⚠️ พี่ปลาระบบขัดข้องเล็กน้อยครับ (Error: {str(e)})"

print('✅ ตั้งค่า Logic พี่ปลาเรียบร้อย!')

📦 กำลังรวมข้อมูลเข้าสู่ Vector Store...
✅ ตั้งค่า Logic พี่ปลาเรียบร้อย!


## 6 :เชื่อมต่อ Frontend ด้วย Ngrok

In [ ]:
import threading
import time

# ── ตั้งค่า ngrok ──
ngrok.set_auth_token(ngrok_token)

# ── Start ngrok Tunnel ──
public_url = ngrok.connect(8000)
print(f"🚀 Public URL: {public_url}")

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# ── Start Uvicorn in a separate thread ──
thread = threading.Thread(target=run_uvicorn)
thread.daemon = True # Allow the main program to exit even if the thread is still running
thread.start()

print("💡 Uvicorn server is running in the background. You can now use the Public URL.")
# Give the server a moment to start up
time.sleep(2)


🚀 Public URL: NgrokTunnel: "https://apportionable-stealthful-gaylord.ngrok-free.dev" -> "http://localhost:8000"
💡 Uvicorn server is running in the background. You can now use the Public URL.


INFO:     Started server process [10040]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [ ]:
# รันเซลล์นี้เมื่อต้องการปิด Server และ Ngrok

# ตรวจสอบว่า uvicorn_server_obj ถูกตั้งค่าไว้หรือไม่ก่อนสั่งปิด
if 'uvicorn_server_obj' in globals() and uvicorn_server_obj is not None:
    uvicorn_server_obj.should_exit = True
    print('Signal sent to Uvicorn server to shut down.')
else:
    print('Uvicorn server object not found or not running.')

# ตรวจสอบว่า ngrok_tunnel_obj ถูกตั้งค่าไว้หรือไม่ก่อนสั่งปิด
if 'ngrok_tunnel_obj' in globals() and ngrok_tunnel_obj is not None:
    ngrok_tunnel_obj.disconnect()
    print('Ngrok tunnel disconnected.')
else:
    print('Ngrok tunnel object not found or not active.')

print('🛑 ปิดการทำงานของ Server และ ngrok เรียบร้อยแล้ว!')


Uvicorn server object not found or not running.
Ngrok tunnel object not found or not active.
🛑 ปิดการทำงานของ Server และ ngrok เรียบร้อยแล้ว!


## 7 : ทดสอบแบบ Interactive (Command Line)

In [8]:
# 🐟 ทดสอบ interactive ใน Colab (Aqua Bot V2.1)
print('🐟 สวัสดีครับ! พี่ปลาพร้อมให้ข้อมูล **สถิติปลาเศรษฐกิจปี 2563 - 2566** แล้วครับ')
print('💡 ตัวอย่างคำถาม:')
print('   - "ราคาปลานิลปี 65 เป็นอย่างไร?"')
print('   - "จังหวัดไหนผลผลิตปลาดุกเยอะที่สุดในปี 66?"')
print('   - "เปรียบเทียบราคาปลาตะเพียนปี 64 กับ 65"')
print('⚠️ *ณ ตอนนี้พี่ปลาเน้นข้อมูลสถิติและราคา ไม่ครอบคลุมเทคนิคการเลี้ยงและโรคปลานะครับ*')
print('พิมพ์ "exit" เพื่อจบการสนทนา')
print('-' * 60)

chat_history = []

while True:
    user_input = input('\n❓ เกษตรกร: ').strip()

    # 1. เช็กการออกจากโปรแกรม
    if user_input.lower() in ['exit', 'ออก', 'quit', 'bye']:
        print('\n🐟 พี่ปลา: 👋 ขอให้ได้ราคาดีๆ ผลผลิตปังๆ นะครับ! ขอบคุณที่ใช้บริการพี่ปลาครับ')
        break

    # 2. เช็กกรณีค่าว่าง
    if not user_input:
        continue

    # 3. ส่งคำถามเข้าฟังก์ชัน handle_query (ที่รวม Logic Refusal ไว้แล้ว)
    print('\n🐟 พี่ปลา กำลังหาข้อมูลให้สักครู่นะครับ...')
    answer = handle_query(user_input, chat_history)

    print(f'🐟 พี่ปลา: {answer}')
    chat_history.append((user_input, answer)) # Add to chat history
    print('-' * 40)


🐟 สวัสดีครับ! พี่ปลาพร้อมให้ข้อมูล **สถิติปลาเศรษฐกิจปี 2563 - 2566** แล้วครับ
💡 ตัวอย่างคำถาม:
   - "ราคาปลานิลปี 65 เป็นอย่างไร?"
   - "จังหวัดไหนผลผลิตปลาดุกเยอะที่สุดในปี 66?"
   - "เปรียบเทียบราคาปลาตะเพียนปี 64 กับ 65"
⚠️ *ณ ตอนนี้พี่ปลาเน้นข้อมูลสถิติและราคา ไม่ครอบคลุมเทคนิคการเลี้ยงและโรคปลานะครับ*
พิมพ์ "exit" เพื่อจบการสนทนา
------------------------------------------------------------

❓ เกษตรกร: ปลาดุกเลี้ยงยังไง

🐟 พี่ปลา กำลังหาข้อมูลให้สักครู่นะครับ...
🐟 พี่ปลา: สวัสดีครับ! พี่ปลา มีประสบการณ์ในการเลี้ยงปลาดุกมาอย่างยาวนานครับ ปลาดุกเป็นปลาที่มีลักษณะพิเศษ เช่น ไม่มีเกล็ด รูปร่างเรียวยาว มีหนวด 4 คู่ และสามารถทนอยู่บนบกได้นานกว่าปลาชนิดอื่นๆ

ในการเลี้ยงปลาดุก คุณควรให้อาหารที่มีเนื้อสัตว์เป็นหลัก แต่สามารถฝึกให้กินอาหารพืชได้เช่นกัน นอกจากนี้ คุณควรตรวจสอบปลาดุกอย่างสม่ำเสมอเพื่อป้องกันโรคจากหมัดปลา ซึ่งเป็นโรคที่สามารถทำให้ปลาตายได้ภายในเวลา 3-4 ชั่วโมง

หากคุณมีคำถามเพิ่มเติมเกี่ยวกับการเลี้ยงปลาดุกหรือต้องการคำแนะนำเพิ่มเติม พี่ปลาอยู่ที่นี่เพื่อช่วยเหลือครับ
-------

KeyboardInterrupt: Interrupted by user